# E14 · Use a small SWRL rule deliberately

**Outcome:** Write a positive relational rule, execute it on named source records, and choose when SWRL is appropriate.

**Time:** about 50 minutes. Run cells in order. Edit the exercise cell after completing the walkthrough.

SWRL expresses an implication: if every body atom holds for a variable binding, the head follows. It is a W3C Member Submission, not an OWL 2 profile or a W3C Recommendation. Unrestricted OWL plus SWRL loses OWL DL's general decidability guarantee. A rule whose head variables appear in the body is range-restricted, but that condition alone is not DL-safety. DL-safe approaches restrict rule variables to an explicitly controlled domain, commonly named individuals, under engine-specific semantics.

This lesson uses the pinned HermiT integration on an isolated rule ontology populated from the same 60 source records. It tests a positive join that relates a respiratory-coded record to the patient reached through its encounter. The result routes information for review; it is not a clinical diagnosis. The rule module remains outside the core OWL 2 DL profile claim. Ordinary OWL class guards do not by themselves prove formal DL-safety, and success on this dataset does not establish support for every SWRL construct.

Prefer ordinary OWL axioms when a subclass, class restriction, inverse or property chain states the intended meaning. Consider SWRL when a supported positive rule needs a variable join that the chosen OWL modeling pattern does not express conveniently and its scope can be controlled. Use SHACL for required fields and data rejection. Use SPARQL for dataset-relative absence, aggregation and explicit transformations. Use Python or an application service for orchestration, external calls, state changes and scheduled actions.

SWRL is not a general workflow engine. Standard rules are monotonic and do not provide ordinary closed-world negation-as-failure, deletion, arbitrary aggregate queries or safe external side effects. Built-ins such as arithmetic and comparison vary by reasoner. Do not assume the greaterThanOrEqual built-in works merely because a rule string parses. This tested rule uses no built-ins. After removing supporting facts, rebuild the rule ontology from source to avoid stale inferred properties.

In [1]:
from pathlib import Path
import sys, json
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "ontology_lab").is_dir():
        sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError("Open this notebook from the extracted course folder.")
from ontology_lab import *
print("Course:", ROOT.name, "| source rows:", len(rows()))

Course: enterprise_ontology_tutorial | source rows: 60


## Declare the rule vocabulary and load real records

In [2]:
from owlready2 import World, Thing, ObjectProperty, Imp, sync_reasoner
world=World()
onto=world.get_ontology('https://example.org/health/rules/')
with onto:
    class EncounterRecord(Thing): pass
    class HospitalEncounter(Thing): pass
    class Patient(Thing): pass
    class Code(Thing): pass
    class RespiratoryCode(Code): pass
    class documents(ObjectProperty): pass
    class hasPatient(ObjectProperty): pass
    class primaryCode(ObjectProperty): pass
    class respiratoryReviewFor(ObjectProperty): pass
    for row in rows():
        record=EncounterRecord('record_'+row['encounter_id'])
        event=HospitalEncounter('encounter_'+row['encounter_id'])
        person=Patient('patient_'+row['patient_nbr'])
        cls=RespiratoryCode if row['diag_1']=='493' else Code
        concept=cls('code_'+row['diag_1'])
        record.documents=[event]
        event.hasPatient=[person]
        record.primaryCode=[concept]
assert sum(len(x.respiratoryReviewFor) for x in EncounterRecord.instances())==0

## Add the positive join rule and run its supported engine

In [3]:
with onto:
    rule=Imp()
    rule.set_as_rule(
        'EncounterRecord(?r), documents(?r, ?e), '
        'HospitalEncounter(?e), hasPatient(?e, ?p), '
        'Patient(?p), primaryCode(?r, ?c), RespiratoryCode(?c) '
        '-> respiratoryReviewFor(?r, ?p)')
print(rule)
sync_reasoner([onto],infer_property_values=True,debug=0)
actual={(r.name.removeprefix('record_'),p.name.removeprefix('patient_'))
        for r in EncounterRecord.instances() for p in r.respiratoryReviewFor}
expected={(r['encounter_id'],r['patient_nbr']) for r in rows() if r['diag_1']=='493'}
assert actual==expected and len(actual)==20
display(sorted(actual)[:5])

EncounterRecord(?r), documents(?r, ?e), HospitalEncounter(?e), hasPatient(?e, ?p), Patient(?p), primaryCode(?r, ?c), RespiratoryCode(?c) -> respiratoryReviewFor(?r, ?p)


[('10555854', '2571498'),
 ('12846246', '4634397'),
 ('13056282', '278892'),
 ('14964918', '3192903'),
 ('15996702', '1133793')]

## Your turn

Choose a mechanism for each requirement: required primary-code field, count by code family, supported positive relational join, and a basic class inclusion. Return a dictionary.

Replace `answer = None` with your code. A skipped exercise is reported as incomplete; it is not a pass.

In [4]:
answer = None  # Write your solution here

In [5]:
learner_check(answer, lambda x:x=={'required_field':'SHACL','aggregate':'SPARQL','positive_join':'SWRL','class_inclusion':'OWL'}, 'The simplest mechanism with the required semantics is usually easiest to govern.')

Exercise not completed. The simplest mechanism with the required semantics is usually easiest to govern.
Out[0]: False


## Explain your model

Why does this positive rule not prove anything about records without a respiratory source code? What must be rebuilt when a supporting edge is removed?

Write a short answer below. Check the relevant chapter in the book before promoting a model change.

**My explanation:** _Write your explanation here._